# 01. Dataset Exploration

CCTV 영상 데이터의 기본 특성(해상도, FPS, 길이)과 YOLO 검출 결과 분포를 확인하고,
위험구역(ROI) 좌표와 근접위험 임계값(`distance_px`)을 정하는 근거를 만든다.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

with open("../configs/config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
VIDEO = os.path.join("..", cfg["video"]["source"])

## 1. 영상 메타데이터

In [ ]:
cap = cv2.VideoCapture(VIDEO)
meta = {
    "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "fps": cap.get(cv2.CAP_PROP_FPS),
    "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
}
meta["duration_sec"] = meta["frames"] / meta["fps"]
cap.release()
meta

## 2. 샘플 프레임 + 위험구역 오버레이
좌표가 실제 바닥 위치와 맞는지 눈으로 확인한다.

In [ ]:
from src.risk.zone import Zone

cap = cv2.VideoCapture(VIDEO)
ok, frame = cap.read(); cap.release()
w = cfg["video"]["resize_width"]
if w:
    frame = cv2.resize(frame, (w, int(frame.shape[0] * w / frame.shape[1])))
for z in cfg["risk"]["zones"]:
    Zone.from_config(z).draw(frame)
plt.figure(figsize=(12, 7)); plt.imshow(frame[..., ::-1]); plt.axis("off");

## 3. 검출 통계 (일정 간격 샘플링)

In [ ]:
from src.detection.detector import YOLODetector
from src.video.video_source import VideoSource

det = YOLODetector(cfg["detection"]["model"].replace("models/", "../models/"),
                   class_map=cfg["detection"]["class_map"], device=cfg["detection"]["device"])
rows = []
with VideoSource(VIDEO, cfg["video"]["resize_width"]) as src:
    for i, f in enumerate(src):
        if i % 10:
            continue
        for d in det.detect(f):
            x1, y1, x2, y2 = d.bbox
            rows.append(dict(frame=i, category=d.category, score=d.score,
                             w=x2 - x1, h=y2 - y1, fx=(x1 + x2) / 2, fy=y2))
df = pd.DataFrame(rows)
df.groupby("category").agg(count=("score", "size"), mean_score=("score", "mean"),
                           mean_h=("h", "mean"))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for cat, g in df.groupby("category"):
    ax[0].hist(g.score, bins=30, alpha=.6, label=cat)
    ax[1].hist(g.h, bins=30, alpha=.6, label=cat)
    ax[2].scatter(g.fx, g.fy, s=4, alpha=.4, label=cat)
ax[0].set_title("confidence"); ax[1].set_title("bbox height (px)")
ax[2].set_title("foot point 분포"); ax[2].invert_yaxis()
for a in ax: a.legend()
plt.tight_layout()

## 4. 임계값 설정 근거

- 사람 bbox 평균 높이(px)는 약 1.7m 에 해당 → `px_per_meter ≈ mean_h / 1.7`
- 안전거리 1.5m 기준이면 `distance_px ≈ 1.5 * px_per_meter`
- 단, 원근 때문에 화면 아래쪽(가까운 곳)일수록 값이 커짐 → 추후 호모그래피 보정 필요

In [ ]:
px_per_m = df[df.category == "person"].h.mean() / 1.7
print(f"px/m ≈ {px_per_m:.1f},  1.5m 안전거리 ≈ {1.5 * px_per_m:.0f}px")